# VGG Cosmology Bias Check

This notebook summarizes the VGG-based cosmology encoder and uses it to check whether conditional HI diffusion samples preserve the requested cosmology.

The poster-level message is simple:

```text
requested cosmology -> conditional diffusion -> generated HI field -> VGG encoder -> recovered cosmology
```

If the recovered parameter follows the requested parameter, then the generated field carries the conditioning information. The cleanest current result is for `Omega_m`, so the poster figure below focuses on that one parameter.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Find repository root whether this notebook is run from notebooks/ or from repo root.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / '.git').exists():
    ROOT = ROOT.parent

RESULT_ROOT = ROOT / 'results' / 'nf_conditional_bias_probe'
ENCODER_DIR = RESULT_ROOT / 'encoder'
CALIB_DIR = RESULT_ROOT / 'calibration_vgg'
POSTER_DIR = CALIB_DIR / 'poster'
POSTER_DIR.mkdir(parents=True, exist_ok=True)

BEST_VGG_ENCODER = ENCODER_DIR / 'vgg_mlp_big_avgmax.npz'
BEST_VGG_MODEL = ENCODER_DIR / 'vgg_mlp_big_avgmax.pkl'

REAL_METRICS_PATH = ENCODER_DIR / 'vgg_real_test_metrics.csv'
REAL_PRED_PATH = ENCODER_DIR / 'vgg_real_test_per_cosmology_predictions.csv'
VGG_COMPARISON_PATH = ENCODER_DIR / 'vgg_encoder_r2_comparison.csv'
CALIB_POINTS_PATH = CALIB_DIR / 'bias_probe_per_cosmology_points.csv'
CALIB_SLOPES_PATH = CALIB_DIR / 'bias_probe_regime_slopes.csv'
CALIB_METADATA_PATH = CALIB_DIR / 'bias_probe_eval_metadata.json'

PARAM_ORDER = ['Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2']
PARAM_LABEL = {
    'Omega_m': r'$\Omega_m$',
    'sigma_8': r'$\sigma_8$',
    'A_SN1': r'$A_{SN1}$',
    'A_AGN1': r'$A_{AGN1}$',
    'A_SN2': r'$A_{SN2}$',
    'A_AGN2': r'$A_{AGN2}$',
}
REGIME_LABEL = {
    'memorization': 'Memorization (N=128)',
    'generalization': 'Generalization (N=16,384)',
}
REGIME_COLOR = {
    'memorization': '#D55E00',    # Okabe-Ito vermillion
    'generalization': '#0072B2',  # Okabe-Ito blue
}
REGIME_MARKER = {'memorization': 'o', 'generalization': 's'}

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 1.25,
    'xtick.major.width': 1.15,
    'ytick.major.width': 1.15,
    'xtick.major.size': 5.5,
    'ytick.major.size': 5.5,
})

def read_csv_or_none(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    return None

def read_json_or_none(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text())
    return {}

def show_missing(paths):
    missing = [str(p.relative_to(ROOT)) for p in paths if not p.exists()]
    if missing:
        display(Markdown('**Missing result files in this checkout:**\n\n' + '\n'.join(f'- `{p}`' for p in missing)))
    else:
        display(Markdown('All expected result files are present.'))

print('repo root:', ROOT)
show_missing([REAL_METRICS_PATH, REAL_PRED_PATH, CALIB_POINTS_PATH, CALIB_SLOPES_PATH])

## What The VGG Encoder Does

The encoder is only a diagnostic probe. It is not part of the diffusion model training.

```text
normalized HI slice
-> copy the one HI channel into R, G, B
-> bilinear resize to 224 x 224
-> frozen ImageNet VGG16 convolutional features
-> average + max pooling
-> MLP regression head with hidden layers 1024, 512, 256
-> predicted CAMELS parameters
```

Important safeguards:

- VGG16 is frozen; only the regression head is trained.
- The head is trained on real HI slices from non-held-out simulations only.
- Held-out simulations `900-931` are used for testing and for the diffusion calibration plot.
- The current best encoder is `frozen VGG16 + avg+max pooling + MLP(1024,512,256)`.

In [ ]:
# Recorded summary from the completed Great Lakes VGG tests.
# The live CSVs are preferred when present; this table is here so the notebook remains readable off-cluster.
recorded_vgg_results = pd.DataFrame([
    {'encoder': 'avg+max + MLP(1024,512,256), 65k slices', 'Omega_m': 0.9115, 'sigma_8': 0.7374, 'A_SN1': 0.4544, 'A_AGN1': -0.0063, 'A_SN2': 0.3251, 'A_AGN2': 0.1005},
    {'encoder': 'avg + MLP(1024,512,256), 65k slices',     'Omega_m': 0.9027, 'sigma_8': 0.7349, 'A_SN1': 0.4535, 'A_AGN1': -0.0313, 'A_SN2': 0.2561, 'A_AGN2': 0.0868},
    {'encoder': 'avg+max + MLP(512,256), 32k slices',      'Omega_m': 0.8992, 'sigma_8': 0.6830, 'A_SN1': 0.4189, 'A_AGN1': 0.0144, 'A_SN2': 0.2953, 'A_AGN2': 0.0946},
    {'encoder': 'avg+max + Ridge(alpha=1), 65k slices',    'Omega_m': 0.8977, 'sigma_8': 0.7143, 'A_SN1': 0.4145, 'A_AGN1': -0.0217, 'A_SN2': 0.2862, 'A_AGN2': 0.1418},
])

display(recorded_vgg_results)

## Real Held-Out Sanity Check

Before using the encoder on generated fields, first check that it can recover cosmology from real held-out HI fields. This is a no-diffusion test.

For each held-out simulation, the encoder predicts parameters for many 2D slices. The plotted point is the median prediction over slices, and the vertical error bar is the 16th to 84th percentile spread across those slice-level predictions.

In [ ]:
real_metrics = read_csv_or_none(REAL_METRICS_PATH)
real_pred = read_csv_or_none(REAL_PRED_PATH)
comparison = read_csv_or_none(VGG_COMPARISON_PATH)

if real_metrics is not None:
    real_summary = real_metrics[(real_metrics['split'] == 'test') & (real_metrics['grain'] == 'per_cosmology')].copy()
    real_summary['parameter'] = pd.Categorical(real_summary['parameter'], PARAM_ORDER, ordered=True)
    display(real_summary.sort_values('parameter')[['parameter', 'n', 'mae', 'rmse', 'bias', 'r2']])
else:
    display(Markdown('Live `vgg_real_test_metrics.csv` is not present here. Using the recorded summary above.'))

In [ ]:
def plot_real_encoder_omega(real_pred: pd.DataFrame | None, real_metrics: pd.DataFrame | None, *, out_name='vgg_encoder_omega_real_test_poster.png'):
    if real_pred is None or real_metrics is None:
        display(Markdown('Cannot draw the real-heldout encoder plot because the live VGG prediction CSVs are missing.'))
        return None

    param = 'Omega_m'
    x = real_pred[f'{param}_true'].to_numpy(float)
    y = real_pred[f'{param}_pred_median'].to_numpy(float)
    y16 = real_pred[f'{param}_pred_q16'].to_numpy(float)
    y84 = real_pred[f'{param}_pred_q84'].to_numpy(float)
    yerr = np.vstack([np.maximum(y - y16, 0.0), np.maximum(y84 - y, 0.0)])
    metric = real_metrics[(real_metrics['split'] == 'test') & (real_metrics['grain'] == 'per_cosmology') & (real_metrics['parameter'] == param)]
    r2 = float(metric['r2'].iloc[0]) if len(metric) else np.nan

    lo = min(x.min(), y16.min())
    hi = max(x.max(), y84.max())
    pad = 0.08 * max(hi - lo, 1e-6)
    lo -= pad
    hi += pad

    fig, ax = plt.subplots(figsize=(6.2, 5.2), constrained_layout=True)
    ax.plot([lo, hi], [lo, hi], '--', color='0.25', lw=2.2, label='ideal recovery')
    ax.errorbar(x, y, yerr=yerr, fmt='o', ms=7.5, lw=2.0, capsize=3.2,
                color='#6f4aa8', ecolor='#b8a5d6', markeredgecolor='white', markeredgewidth=0.8)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(r'True $\Omega_m$', fontsize=18)
    ax.set_ylabel(r'VGG-predicted $\Omega_m$', fontsize=18)
    ax.tick_params(labelsize=15)
    ax.set_title(r'VGG encoder sanity check: real held-out HI fields', fontsize=18, pad=12)
    ax.text(0.05, 0.94, rf'$R^2 = {r2:.2f}$', transform=ax.transAxes,
            fontsize=18, fontweight='semibold', ha='left', va='top')
    ax.legend(loc='lower right', frameon=False, fontsize=14)
    ax.grid(False)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

    out = POSTER_DIR / out_name
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

plot_real_encoder_omega(real_pred, real_metrics)

## Generated-Field Calibration

For the diffusion test, the input is a held-out cosmology `theta_in`. For each held-out cosmology, the model generates multiple HI samples with different noise seeds but the same requested cosmology.

For one parameter, the plotted point is:

```text
median recovered theta over generated samples at fixed theta_in
```

The error bar is:

```text
16th to 84th percentile spread of recovered theta over those generated samples
```

The fitted slope is the headline number:

```text
slope near 0: generated fields mostly ignore the requested cosmology
slope near 1: generated fields track the requested cosmology
```

The poster figure below only shows `Omega_m`, where the signal is strongest and easiest to explain.

In [ ]:
points = read_csv_or_none(CALIB_POINTS_PATH)
slopes = read_csv_or_none(CALIB_SLOPES_PATH)
metadata = read_json_or_none(CALIB_METADATA_PATH)

if points is not None:
    display(points.head())
    display(points.groupby(['regime', 'dataset_size', 'parameter']).size().reset_index(name='n_points').head(12))
else:
    display(Markdown('Live generated-field calibration points are missing in this checkout.'))

if slopes is not None:
    slope_show = slopes[slopes['parameter'].isin(['Omega_m', 'sigma_8'])].copy()
    display(slope_show[['regime', 'dataset_size', 'parameter', 'slope', 'slope_ci16', 'slope_ci84', 'intercept', 'n_heldout']])

In [ ]:
def plot_omega_calibration_poster(
    points: pd.DataFrame | None,
    slopes: pd.DataFrame | None,
    *,
    out_name='bias_probe_omega_m_best_vgg_poster.png',
    title='Generated HI fields recover the requested matter density',
):
    if points is None or slopes is None:
        display(Markdown('Cannot draw the poster calibration plot because the live calibration CSVs are missing.'))
        return None

    param = 'Omega_m'
    p = points[points['parameter'] == param].copy()
    if p.empty:
        display(Markdown(f'No rows for `{param}`.'))
        return None

    fig, ax = plt.subplots(figsize=(7.6, 5.5), constrained_layout=True)

    lo = float(min(p['theta_in'].min(), p['theta_rec_q16'].min()))
    hi = float(max(p['theta_in'].max(), p['theta_rec_q84'].max()))
    pad = 0.08 * max(hi - lo, 1e-6)
    lo -= pad
    hi += pad
    ax.plot([lo, hi], [lo, hi], '--', color='0.25', lw=2.3, label='ideal recovery', zorder=1)

    legend_handles = []
    legend_labels = []
    for regime in ['memorization', 'generalization']:
        sub = p[p['regime'] == regime].sort_values('theta_in')
        if sub.empty:
            continue
        color = REGIME_COLOR[regime]
        y = sub['theta_rec_median'].to_numpy(float)
        yerr = np.vstack([
            np.maximum(y - sub['theta_rec_q16'].to_numpy(float), 0.0),
            np.maximum(sub['theta_rec_q84'].to_numpy(float) - y, 0.0),
        ])

        slope_row = slopes[(slopes['parameter'] == param) & (slopes['regime'] == regime)]
        slope = float(slope_row['slope'].iloc[0]) if len(slope_row) else np.nan
        intercept = float(slope_row['intercept'].iloc[0]) if len(slope_row) else np.nan
        label = f"{REGIME_LABEL[regime]}"
        if np.isfinite(slope):
            label += f", slope={slope:.2f}"

        handle = ax.errorbar(
            sub['theta_in'], y, yerr=yerr,
            fmt=REGIME_MARKER[regime], ms=8.0, lw=2.0, capsize=3.2, capthick=1.4,
            color=color, ecolor=color, alpha=0.86,
            markeredgecolor='white', markeredgewidth=0.8,
            label=label, zorder=3,
        )
        legend_handles.append(handle)
        legend_labels.append(label)

        if np.isfinite(slope):
            xs = np.array([lo, hi])
            ax.plot(xs, slope * xs + intercept, color=color, lw=3.4, zorder=2)

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(r'Requested $\Omega_m$', fontsize=19)
    ax.set_ylabel(r'Recovered $\Omega_m$', fontsize=19)
    ax.tick_params(labelsize=16)
    ax.set_title(title, fontsize=19, pad=16)
    ax.grid(False)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.16),
              ncol=1, frameon=False, fontsize=13.5, handlelength=2.4)

    ax.text(0.035, 0.955, 'Best encoder: frozen VGG16 + avg+max pooling + MLP(1024,512,256)',
            transform=ax.transAxes, ha='left', va='top', fontsize=12.5, color='0.28')

    out = POSTER_DIR / out_name
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

omega_plot = plot_omega_calibration_poster(points, slopes)

## One-Sentence Poster Caption

Using a frozen VGG16 feature encoder trained only on real non-held-out HI fields, the large-data conditional diffusion model recovers the requested `Omega_m` much better than the small-data model. This supports the interpretation that the generalization-regime model is not just producing plausible fields; it is more faithful to the input cosmology.

## Practical Interpretation

- `Omega_m` is the clearest result and the best poster panel.
- `sigma_8` has some signal but is weaker.
- Feedback parameters are much less reliable from HI alone with this encoder.
- The VGG encoder is a probe, not proof of perfect cosmological calibration. The result should be stated as evidence that conditioning fidelity improves in the higher-data regime.

In [ ]:
# Show the saved poster plot inside the notebook if it was created.
if 'omega_plot' in globals() and omega_plot is not None and Path(omega_plot).exists():
    display(Image(filename=str(omega_plot)))
else:
    display(Markdown('Poster plot not created in this environment because the live calibration CSVs are missing.'))